# 36. HS4 Explanatory-Note Reranking

실험 34의 HS4 직접 분류 기준선을 유지하고, HS 해설서 지식베이스를 별도의 근거 검색 채널로 추가합니다. 해설서의 호별 분류기준·포함 범위는 직접 근거 채널로, 제외 규정에서 명시적으로 참조하는 다른 호는 제외 근거 채널로 분리합니다.

- 평가 모집단: 195건, HS4 5개, 제품군 29개
- 평가: Nested Leave-One-Product-Family-Out
- 1차 지표: HS4 Macro-F1 / 2차 지표: HS4 Accuracy
- 기준선: 실험 34의 SBERT Product + Ruling RRF(top3_mean, w=1)
- 개선 모델: 기준선 + 해설서 직접 근거 + 제외 근거의 RRF 결합
- Fold 내부 튜닝만 허용하며, 홀드아웃 제품군은 설정 선택에 사용하지 않음
- Google Drive 미사용

## 실험 설계 및 주의사항

해설서 지식베이스의 HS4별 문서를 검색해 후보 호별 근거 순위를 만듭니다. 해설서의 HS 코드 표기 자체가 임베딩에 누출되지 않도록 메타데이터의 HS 코드·규칙 ID는 임베딩 문장에서 제거합니다. 다만 해설서 원문에 등장하는 다른 호 번호와 법적 문맥은 근거의 일부이므로 유지합니다.

제외 근거는 규칙유형에 '제외'가 포함된 문서에서 명시적으로 참조된 HS4 코드에만 연결합니다. 이는 자동 품목분류 규칙이 아니라 보조 검색 신호이며, 결과 CSV에 근거 문서 ID와 원문을 함께 저장해 추적할 수 있게 합니다.

내부 LOPO에서 선택하는 조합은 직접 근거 집계 방식·제외 근거 집계 방식(max/top-3 mean)과 두 채널의 RRF 가중치입니다. 기준선 모델의 점수는 실험 34와 같은 방식으로 고정합니다.

## 0. 환경 설정

In [ ]:
!pip -q install -U sentence-transformers scipy
import re, json, zipfile, requests
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
WORK=Path('/content/36_hs4_explanatory_note_results')
WORK.mkdir(parents=True,exist_ok=True)
BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/'
print(WORK)

## 1. 실험 29와 동일한 데이터 파이프라인

In [ ]:
src=requests.get(BASE+'29_discriminative_features_experiment.ipynb').json()
for i in [3,5,7]: exec(''.join(src['cells'][i]['source']),globals())
print('평가 표본:',len(eval_df),'| HS6:',len(labels),'| 제품군:',eval_df.product_family.nunique(),'| HS4:',eval_df.hs4.nunique())
assert len(eval_df)==195 and len(labels)==11 and eval_df.product_family.nunique()==29
hs4_labels=sorted(eval_df.hs4.astype(str).unique())
print('HS4 labels:',hs4_labels)
display(eval_df.groupby('hs4').agg(n=('hs4','size'),families=('product_family','nunique')))

## 2. 해설서 지식베이스 정리 및 근거 채널 생성

In [ ]:
kb=pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
for c in ['knowledge_id','hs_code','rule_type','document_text','source_sheet','source_page_or_line']:
    if c not in kb: kb[c]=''
    kb[c]=kb[c].astype(str).str.strip()
kb['hs_digits']=kb.hs_code.map(lambda x:re.sub(r'\D','',x))
def clean_kb_text(x):
    lines=[]
    for line in str(x).splitlines():
        if re.match(r'^(규칙_ID|HS코드|자료출처|류|적용범위)\s*:',line.strip()): continue
        lines.append(line.strip())
    return ' '.join(x for x in lines if x)
kb['embedding_text']=kb.document_text.map(clean_kb_text)
direct_rows=[]
exclusion_rows=[]
for _,r in kb.iterrows():
    typ=r.rule_type
    own=r.hs_digits[:4]
    if own in hs4_labels and r.embedding_text and '제외' not in typ:
        direct_rows.append({**r.to_dict(),'candidate_hs4':own,'channel':'direct'})
    if '제외' in typ and r.embedding_text:
        refs=sorted(set(re.findall(r'(?:제\s*)?(\d{4})\s*호',r.document_text)))
        for code in refs:
            h=code[:4]
            if h in hs4_labels:
                exclusion_rows.append({**r.to_dict(),'candidate_hs4':h,'channel':'exclusion'})
direct_docs=pd.DataFrame(direct_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
exclusion_docs=pd.DataFrame(exclusion_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
print('전체 KB:',len(kb),'| 직접 근거 문서:',len(direct_docs),'| 제외 근거 연결:',len(exclusion_docs))
display(pd.concat([direct_docs.groupby('candidate_hs4').size().rename('direct_docs'),exclusion_docs.groupby('candidate_hs4').size().rename('exclusion_links')],axis=1).fillna(0).astype(int).reindex(hs4_labels).fillna(0))
display(exclusion_docs[['knowledge_id','candidate_hs4','rule_type','source_page_or_line','document_text']])

## 3. 임베딩 생성 및 캐시

In [ ]:
CACHE=WORK/'embeddings_36.npz'
if CACHE.exists():
    z=np.load(CACHE,allow_pickle=True)
    query_emb,product_emb,ruling_emb,kb_direct_emb,kb_exclusion_emb=z['query_emb'],z['product_emb'],z['ruling_emb'],z['kb_direct_emb'],z['kb_exclusion_emb']
else:
    sbert=SentenceTransformer('intfloat/multilingual-e5-base')
    def encode_query(x): return sbert.encode(['query: '+str(v) for v in x],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    def encode_passage(x): return sbert.encode(['passage: '+str(v) for v in x],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    query_emb=encode_query(eval_df.query_text.tolist())
    product_emb=encode_passage(eval_df.query_text.tolist())
    ruling_emb=encode_passage(eval_df.ruling_text.tolist())
    kb_direct_emb=encode_passage(direct_docs.embedding_text.tolist()) if len(direct_docs) else np.empty((0,768),dtype=float)
    kb_exclusion_emb=encode_passage(exclusion_docs.embedding_text.tolist()) if len(exclusion_docs) else np.empty((0,768),dtype=float)
    np.savez_compressed(CACHE,query_emb=query_emb,product_emb=product_emb,ruling_emb=ruling_emb,kb_direct_emb=kb_direct_emb,kb_exclusion_emb=kb_exclusion_emb)
print(query_emb.shape,product_emb.shape,ruling_emb.shape,kb_direct_emb.shape,kb_exclusion_emb.shape)

## 4. HS4 순위 및 해설서 검색 함수

In [ ]:
def aggregate_hs4(scores,codes,method='top3_mean'):
    d=pd.DataFrame({'hs4':np.asarray(codes).astype(str),'score':np.asarray(scores,dtype=float)})
    if method=='max': out=d.groupby('hs4').score.max()
    elif method=='mean': out=d.groupby('hs4').score.mean()
    else: out=d.groupby('hs4').score.apply(lambda x:x.nlargest(min(3,len(x))).mean())
    return out.sort_values(ascending=False)
def rrf_two(a,b,k=60,w=1.0):
    pa={str(c):i+1 for i,c in enumerate(a.index)}
    pb={str(c):i+1 for i,c in enumerate(b.index)}
    return pd.Series({c:1/(k+pa.get(c,99999))+w/(k+pb.get(c,99999)) for c in hs4_labels}).sort_values(ascending=False)
def rank_kb(scores,docs,method):
    if len(docs)==0: return pd.Series({c:-1.0 for c in hs4_labels}).sort_values(ascending=False)
    vals={}
    for c in hs4_labels:
        x=np.asarray(scores)[docs.candidate_hs4.astype(str).to_numpy()==c]
        if len(x)==0: vals[c]=-1.0
        elif method=='max': vals[c]=float(np.max(x))
        elif method=='mean': vals[c]=float(np.mean(x))
        else: vals[c]=float(np.mean(np.sort(x)[-min(3,len(x)):]))
    return pd.Series(vals).sort_values(ascending=False)
def rrf_three(base,direct,exclusion,wd,we,k=60):
    p0={str(c):i+1 for i,c in enumerate(base.index)}
    p1={str(c):i+1 for i,c in enumerate(direct.index)}
    p2={str(c):i+1 for i,c in enumerate(exclusion.index)}
    return pd.Series({c:1/(k+p0.get(c,99999))+wd/(k+p1.get(c,99999))+we/(k+p2.get(c,99999)) for c in hs4_labels}).sort_values(ascending=False)
def baseline_rank(train_idx,q):
    codes=eval_df.iloc[train_idx].hs4.astype(str).to_numpy()
    rp=aggregate_hs4(product_emb[train_idx]@query_emb[q],codes,'top3_mean')
    rr=aggregate_hs4(ruling_emb[train_idx]@query_emb[q],codes,'top3_mean')
    return rrf_two(rp,rr,w=1.0)
direct_codes=direct_docs.candidate_hs4.astype(str).to_numpy()
exclusion_codes=exclusion_docs.candidate_hs4.astype(str).to_numpy()
splits=list(LeaveOneGroupOut().split(eval_df,eval_df.hs4,groups=eval_df.product_family))
print('Outer LOPO:',len(splits))

## 5. Fold 내부 튜닝 및 외부 예측

In [ ]:
FOLD_START=1
FOLD_END=5
METHODS=['max','top3_mean']
WEIGHTS=[0.0,0.5,1.0,2.0]
CONFIGS=[(md,me,wd,we) for md in METHODS for me in METHODS for wd in WEIGHTS for we in WEIGHTS]
def kb_ranks_for_query(q):
    ds=kb_direct_emb@query_emb[q] if len(direct_docs) else np.array([])
    es=kb_exclusion_emb@query_emb[q] if len(exclusion_docs) else np.array([])
    return ({m:rank_kb(ds,direct_docs,m) for m in METHODS},{m:rank_kb(es,exclusion_docs,m) for m in METHODS})
kb_rank_cache={i:kb_ranks_for_query(i) for i in range(len(eval_df))}
def predict_config(base,kbr,cfg):
    md,me,wd,we=cfg
    dr,er=kbr
    return str(rrf_three(base,dr[md],er[me],wd,we).index[0])
def tune_inner(outer_train):
    inner_df=eval_df.iloc[outer_train].reset_index(drop=True)
    inner_splits=list(LeaveOneGroupOut().split(inner_df,inner_df.hs4,groups=inner_df.product_family))
    records={c:[] for c in CONFIGS}
    used=0
    for itr,iva in inner_splits:
        gtr=outer_train[itr]
        gva=outer_train[iva]
        train_labels=set(eval_df.iloc[gtr].hs4.astype(str))
        valid_labels=set(eval_df.iloc[gva].hs4.astype(str))
        if not valid_labels.issubset(train_labels): continue
        used+=1
        for q in gva:
            base=baseline_rank(gtr,q)
            kbr=kb_rank_cache[q]
            true=str(eval_df.iloc[q].hs4)
            for cfg in CONFIGS:
                records[cfg].append((true,predict_config(base,kbr,cfg)))
    if used==0: raise ValueError('유효한 Inner LOPO Fold가 없습니다.')
    scored=[]
    for cfg,vals in records.items():
        yt=[x[0] for x in vals]; yp=[x[1] for x in vals]
        scored.append((f1_score(yt,yp,labels=hs4_labels,average='macro',zero_division=0),accuracy_score(yt,yp),cfg))
    scored.sort(key=lambda x:(x[0],x[1]),reverse=True)
    return scored[0][2],scored[0][0],scored[0][1],used
def fold_path(f): return WORK/f'fold_{f:02d}.csv'
for fold,(tr,te) in enumerate(splits,1):
    if not FOLD_START<=fold<=FOLD_END: continue
    if fold_path(fold).exists(): print('Fold',fold,'완료 - skip'); continue
    train_codes=set(eval_df.iloc[tr].hs4.astype(str))
    test_codes=set(eval_df.iloc[te].hs4.astype(str))
    if not test_codes.issubset(train_codes): raise ValueError(f'Fold {fold}: 학습 데이터에 없는 HS4')
    cfg,inner_f1,inner_acc,inner_folds=tune_inner(tr)
    md,me,wd,we=cfg
    rows=[]
    for q in te:
        base=baseline_rank(tr,q)
        kbr=kb_rank_cache[q]
        tuned=rrf_three(base,kbr[0][md],kbr[1][me],wd,we)
        for model,rank in [('Baseline_34_RRF',base),('HS_Explanatory_Note_RRF',tuned)]:
            rows.append({'row_index':int(q),'fold':fold,'heldout_family':eval_df.iloc[q].product_family,'true_hs4':str(eval_df.iloc[q].hs4),'model':model,'predicted_hs4':str(rank.index[0]),'top3_hs4':'|'.join(rank.index[:3].astype(str)),'selected_direct_method':md,'selected_exclusion_method':me,'selected_direct_weight':wd,'selected_exclusion_weight':we,'inner_macro_f1':inner_f1,'inner_accuracy':inner_acc})
    pd.DataFrame(rows).to_csv(fold_path(fold),index=False,encoding='utf-8-sig')
    print(f'Fold {fold}/{len(splits)} | direct={md}, exclusion={me}, weights=({wd},{we}) | inner F1={inner_f1:.4f}')
print('현재 실행 구간 완료')

## 6. 진행률 및 중간 백업

In [ ]:
done=sorted(int(p.stem.split('_')[1]) for p in WORK.glob('fold_*.csv'))
missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)} | 누락: {missing}')
BACKUP='/content/36_fold_backup.zip'
with zipfile.ZipFile(BACKUP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in WORK.glob('fold_*.csv'): z.write(p,p.name)
    if CACHE.exists(): z.write(CACHE,CACHE.name)
from google.colab import files
files.download(BACKUP)

## 7. 중간 백업 복구 (새 런타임에서만 실행)

In [ ]:
from google.colab import files
uploaded=files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall(WORK)
        print('복구 완료:',name)

## 8. 통합 평가 및 클래스별 진단

In [ ]:
files_list=sorted(WORK.glob('fold_*.csv'))
done=sorted(int(p.stem.split('_')[1]) for p in files_list)
missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)} | 누락: {missing}')
if missing: raise ValueError('미완료 Fold가 있습니다.')
pred=pd.concat([pd.read_csv(p) for p in files_list],ignore_index=True)
if pred.duplicated(['row_index','model']).any(): raise ValueError('중복 예측 존재')
def metric(g):
    yt=g.true_hs4.astype(str); yp=g.predicted_hs4.astype(str)
    return pd.Series({'n':len(g),'HS4_Macro_F1':f1_score(yt,yp,labels=hs4_labels,average='macro',zero_division=0),'HS4_Accuracy':accuracy_score(yt,yp)})
summary=pred.groupby('model').apply(metric,include_groups=False).reset_index().sort_values(['HS4_Macro_F1','HS4_Accuracy'],ascending=False)
class_rows=[]
for model,g in pred.groupby('model'):
    p,r,f,s=precision_recall_fscore_support(g.true_hs4.astype(str),g.predicted_hs4.astype(str),labels=hs4_labels,zero_division=0)
    for c,pv,rv,fv,sv in zip(hs4_labels,p,r,f,s): class_rows.append({'model':model,'HS4':c,'support':int(sv),'precision':pv,'recall':rv,'F1':fv})
class_metrics=pd.DataFrame(class_rows)
display(summary.round(4)); display(class_metrics.sort_values(['model','HS4']).round(4))
for model,g in pred.groupby('model'):
    display(pd.DataFrame(confusion_matrix(g.true_hs4.astype(str),g.predicted_hs4.astype(str),labels=hs4_labels),index=hs4_labels,columns=hs4_labels))
OUT=WORK/'final'; OUT.mkdir(exist_ok=True)
pred.to_csv(OUT/'36_oof_predictions.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'36_model_comparison.csv',index=False,encoding='utf-8-sig')
class_metrics.to_csv(OUT/'36_hs4_class_metrics.csv',index=False,encoding='utf-8-sig')
for model,g in pred.groupby('model'):
    pd.crosstab(g.true_hs4,g.predicted_hs4).to_csv(OUT/f'36_confusion_{model}.csv',encoding='utf-8-sig')
display(pd.concat([direct_docs.assign(channel='direct'),exclusion_docs.assign(channel='exclusion')])[['knowledge_id','candidate_hs4','channel','rule_type','source_page_or_line','document_text']])
pd.concat([direct_docs.assign(channel='direct'),exclusion_docs.assign(channel='exclusion')]).to_csv(OUT/'36_kb_evidence_inventory.csv',index=False,encoding='utf-8-sig')
with zipfile.ZipFile('/content/36_hs4_explanatory_note_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*.csv'): z.write(p,arcname=p.name)
print('저장 완료: /content/36_hs4_explanatory_note_results.zip')

## 결과 해석 시 주의사항

- 실험 34와 같은 195건·5개 HS4·29개 제품군 평가 모집단을 유지합니다.
- 해설서 지식베이스는 고정된 외부 근거로 사용하며, 제품 정답 라벨을 추가 학습하지 않습니다.
- 제외 규정의 참조 호를 후보 HS4에 연결한 것은 검색 신호일 뿐 법적 판정 자동화가 아닙니다.
- 특정 HS4에 직접 근거 문서가 적거나 없으면 해당 채널은 불리할 수 있습니다. 특히 3401은 제33류 해설서의 제외 문구를 통해서만 근거가 연결될 수 있으므로 별도 확인이 필요합니다.
- Inner LOPO에서 선택한 설정과 Outer LOPO 성능은 구분해서 해석합니다.
- Macro-F1을 우선하고 Accuracy와 클래스별 재현율을 함께 확인합니다.